# NLP Final Project: Mitigation of Dataset Artifacts in NLI

## Part I: Diagnostic Analysis (Code Overview)
This section implements three diagnostic tests to prove the SNLI dataset is compromised.

* **N-Gram Artifact Discovery:** A script that tokenizes the hypothesis sentences and calculates the conditional probability $P(\text{Label} | \text{Word})$. This identifies words like "nobody" that act as ground-truth labels.
* **Hypothesis-Only Baseline:** A training loop that deliberately masks the premise input. High accuracy here serves as a proof-of-concept that the model is "cheating" by using the artifacts identified above.
* **Dataset Cartography:** A logging utility that tracks the training dynamics (confidence and variability) of every example over 6 epochs to visually map the "Easy-to-Learn" vs. "Ambiguous" regions of the data.

## Part II: Mitigation Pipeline
The code implements a two-stage data curation strategy:

1.  **Ambiguity Filtering:**
    * Loads the cartography logs.
    * **Filter Logic:** Drops examples with `variability < threshold` (the easy-to-learn instances).
    * **Hard Negative Mining:** Applies a Jaccard Similarity function to find pairs with $>50\%$ word overlap but different labels.

2.  **Multilingual Augmentation:**
    * **Translation:** Uses `MarianMT` to translate the filtered data (English $\rightarrow$ Spanish $\rightarrow$ English).
    * **Semantic Check:** Uses `SentenceTransformer` (SBERT) to encode the original vs. back-translated sentences.
    * **Quality Logic:** Discards the new sentence if Cosine Similarity is $< 0.85$ (too different) or if it is identical strings (no diversity).

## Final Results (ANLI Benchmark)
Comparing the robustness of models trained on the Full vs. Curated datasets.

| Model | Training Data | ANLI Accuracy (Round 1) |
| :--- | :--- | :--- |
| **Baseline** | Full SNLI (550k) | **32.6%** |
| **Ambiguous Contrast** | Filtered Only (8.7k) | **34.5%** |
| **Augmented Ambiguous** | Filtered + Translated (16.2k) | **34.9%** |

In [6]:
import os

# Disable external logging tool (WandB) to prevent login prompts
os.environ["WANDB_DISABLED"] = "true"

!pip install transformers datasets evaluate scipy scikit-learn # Install dependencies

# Verify we are in the right place. We should see 'run.py' and 'helpers.py' in the output
!ls


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
LICENSE             experiments.ipynb   report.pdf          run.py
README.md           helpers.py          requirements.txt    test_model_baseline


## HuggingFace auth

In [7]:
import os

try:
    # Running in Colab
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
    print("[Env] Token loaded from Colab Secrets")
except ImportError:
    # Not in Colab (PyCharm, local, etc.)
    from dotenv import load_dotenv
    load_dotenv()
    if os.getenv("HF_TOKEN"):
        print("[Env] Token loaded from local .env")
    else:
        print("[Env] Warning: HF_TOKEN not found")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

[Env] Token loaded from local .env


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


## System Sanity Check: Standard Baseline Model

This step executes a minimal training run to establish a control baseline. The objective is to verify that the pipeline is operational and to record the model's initial performance on the standard, artifact-ridden SNLI distribution before any filtering is applied.

### Configuration
* **Dataset:** `snli` (Standard artifact-heavy distribution)
* **Epochs:** `1` (Fast connectivity check)
* **Max Samples:** `100` (Drastically reduced for speed)

In [8]:
%%time

!python3 run.py \
  --do_train \
  --do_eval \
  --task nli \
  --dataset stanfordnlp/snli \
  --output_dir ./test_model_baseline \
  --num_train_epochs 1 \
  --max_train_samples 100 \
  --max_eval_samples 100 \
  --save_steps 0 \
  --eval_strategy epoch \
  --logging_steps 100

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
[HF Engine] Successfully authenticated with Hugging Face Hub
Loading weights: 100%|████████████████████| 199/199 [00:00<00:00, 41823.24it/s]
[transformers] ElectraForSequenceClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.weight         

## Standard Baseline Model Training

This command executes the full training run for the **Standard Baseline Model** (Control Group). This step serves two critical functions in the pipeline:
1.  **Benchmarking:** It establishes the "upper bound" for in-domain accuracy (performance on the standard SNLI test set).
2.  **Data Logging:** It records the training dynamics (logits per epoch) required to calculate Confidence and Variability for the **Dataset Cartography** analysis in Part II.

### Configuration
* **Dataset:** `snli`
* **Output Dir:** `./model_baseline`
* **Epochs:** `6`
* **Eval Strategy:** `epoch` (Logs predictions at the end of every epoch to track variability)

In [ ]:
%%time

!python3 run.py \
  --do_train \
  --do_eval \
  --task nli \
  --dataset stanfordnlp/snli \
  --output_dir ./model_baseline \
  --num_train_epochs 6 \
  --max_length 128 \
  --per_device_train_batch_size 32 \
  --save_steps 0 \
  --eval_strategy epoch \
  --logging_steps 100

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
[HF Engine] Successfully authenticated with Hugging Face Hub
Loading weights: 100%|████████████████████| 199/199 [00:00<00:00, 33699.39it/s]
[transformers] ElectraForSequenceClassification LOAD REPORT from: google/electra-small-discriminator
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight         

## Baseline Performance Evaluation (SNLI & ANLI)

This step executes the critical comparative evaluation of the Standard Baseline Model. We measure performance across two distinct distributions to quantify the Robustness Gap—the difference between how well the model thinks it knows the task versus how well it actually reasons.

### A. In-Domain Accuracy Check (SNLI)
We evaluate the model on the data distribution it was trained on.
* Dataset: `snli`
* Purpose: Establishes the Performance Ceiling. We expect a high score (~90%), confirming the model has successfully memorized the easy artifacts in the standard dataset.

### B. Robustness Check (ANLI)
We evaluate the same model on the Adversarial NLI (ANLI) benchmark.
* Dataset: `anli`
* Purpose: Establishes the Robustness Floor. We anticipate a sharp performance drop (likely approaching random guessing ~33%), empirically proving that the high SNLI score was driven by spurious correlations rather than true semantic understanding.

In [ ]:
%%time

print("\n Evaluating Baseline Model on SNLI")
!python3 run.py \
  --do_eval \
  --task nli \
  --dataset stanfordnlp/snli \
  --model ./model_baseline \
  --output_dir ./eval_model_baseline_snli

print("\n Evaluating Baseline Model on ANLI")
!python3 run.py \
  --do_eval \
  --task nli \
  --dataset facebook/anli \
  --model ./model_baseline \
  --output_dir ./eval_model_baseline_anli

## Cartography Plot Generation

This code block executes the core visualization for the Diagnosis Phase generating the scatter plot for Dataset Cartography to analyze the training dynamics logs generated in the previous step and visualize the entire dataset across the confidence and variability metrics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os
import glob
import json


OUTPUT_DIR = "model_baseline"
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)


def read_dynamics(output_dir):
    """
    Reads all dynamics_epoch_*.jsonl files and aggregates them
    """
    file_pattern = os.path.join(output_dir, "dynamics_epoch_*.jsonl")
    files = sorted(glob.glob(file_pattern))

    if not files:
        print(f"No dynamics files found in {output_dir}. Did training finish?")
        return None

    print(f"Found {len(files)} epoch files. Processing...")

    # Dictionary to store logits for each example across epochs
    # Key: guid, Value: list of {'logits': [], 'gold': int}
    history = {}

    for file_path in files:
        print(f"Reading {os.path.basename(file_path)}...")
        with open(file_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                guid = data['guid']
                logits = data['logits']
                gold = data['gold']

                if guid not in history:
                    history[guid] = {'gold': gold, 'probs': []}


                logits = np.array(logits)
                exp_logits = np.exp(logits - np.max(logits)) # Stable softmax
                probs = exp_logits / exp_logits.sum()

                true_class_prob = probs[gold]
                history[guid]['probs'].append(true_class_prob)

    return history

def compute_metrics(history):
    """
    Calculates confidence and variability for each example
    """
    records = []
    for guid, data in history.items():
        probs = data['probs']


        confidence = np.mean(probs)  # Average prob of true class
        variability = np.std(probs)  # Standard deviation of prob

        records.append({
            'guid': guid,
            'confidence': confidence,
            'variability': variability,
            'gold': data['gold']
        })

    return pd.DataFrame(records)

def plot_cartography(df):
    print("Generating plot...")
    plt.figure(figsize=(10, 8))

    sns.scatterplot(
        data=df,
        x="variability",
        y="confidence",
        hue="confidence",
        palette="coolwarm_r", # Red=Hard (low conf), Blue=Easy (high conf)
        s=10,
        alpha=0.6,
        legend=False
    )

    # Add dividers for the regions (approximate from paper)
    plt.axhline(y=0.5, color='grey', linestyle='--', alpha=0.5)
    plt.axvline(x=0.2, color='grey', linestyle='--', alpha=0.5)

    plt.title("Dataset Cartography: SNLI (ELECTRA-Small)", fontsize=16)
    plt.xlabel("Variability (Standard Deviation)", fontsize=12)
    plt.ylabel("Confidence (Mean Prob of True Label)", fontsize=12)


    plt.text(0.02, 0.9, "Easy-to-Learn", fontsize=12, fontweight='bold', color='darkblue')
    plt.text(0.25, 0.8, "Ambiguous", fontsize=12, fontweight='bold', color='purple')
    plt.text(0.02, 0.1, "Hard-to-Learn", fontsize=12, fontweight='bold', color='darkred')

    save_path = os.path.join(PLOT_DIR, "cartography_map.png")
    plt.savefig(save_path, dpi=300)
    print(f"Plot saved to {save_path}")
    plt.show()

if __name__ == "__main__":
    history = read_dynamics(OUTPUT_DIR)
    if history:
        df = compute_metrics(history)
        print(f"Computed metrics for {len(df)} examples.")
        print(df.head())
        plot_cartography(df)

## Statistical Artifact Analysis & Qualitative Examples

This block performs the quantitative analysis of lexical shortcuts in the hypothesis and provides specific examples to illustrate the findings. The objective is to establish empirical evidence (Part I) that hypothesis n-grams correlate strongly with NLI labels, confirming the model can rely on simple heuristics

In [ ]:
from typing import List
from nltk.stem.snowball import EnglishStemmer
from nltk import word_tokenize
import nltk
from pandas import DataFrame
import random
import datasets
from collections import defaultdict
import pandas as pd

nltk.download('stopwords')
nltk.download('punkt_tab')

def preprocess_text(text: str, stemmer: EnglishStemmer, n_grams: int) -> List[str]:
    if n_grams < 1:
        raise ValueError(f"n_grams must be >= 1, got {n_grams}")

    tokenized_words = word_tokenize(text=text)
    preprocessed_text = []

    for i in range(len(tokenized_words) - n_grams + 1):
        # word = stemmer.stem(word)
        preprocessed_text.append(' '.join(tokenized_words[i:i + n_grams]))

    return preprocessed_text


def run_statistical_test(n_grams: int = 1) -> DataFrame:
    stemmer = EnglishStemmer(ignore_stopwords=True)
    dataset = datasets.load_dataset('snli')
    dataset = dataset.filter(lambda ex: ex['label'] != -1) # remove SNLI examples with no label
    train_data = dataset['train']
    stats = defaultdict(lambda: {'total': 0, 'contradiction': 0, 'entailment': 0, 'neutral': 0})

    for ex in train_data:
        hypothesis = ex['hypothesis']
        label = ex['label']

        for word in preprocess_text(text=hypothesis, stemmer=stemmer, n_grams=n_grams):
            stats[word]['total'] += 1

            match label:
                case 0:
                    stats[word]['entailment'] += 1
                case 1:
                    stats[word]['neutral'] += 1
                case 2:
                    stats[word]['contradiction'] += 1
                case _:
                    print(f'Unknown label {label} in hypothesis {hypothesis}')

    stats = dict(sorted(stats.items(), key=lambda item: item[1]['total'], reverse=True))
    results = []

    for word, stat in stats.items():
        total = stat['total']
        if total < 5: continue

        entailment_count = stat['entailment']
        prob_entailment = (entailment_count / total) * 100

        neutral_count = stat['neutral']
        prob_neutral = (neutral_count / total) * 100

        contradiction_count = stat['contradiction']
        prob_contradiction = (contradiction_count / total) * 100

        if prob_entailment < 50 and prob_neutral < 50 and prob_contradiction < 50:
            continue

        results.append({
            "Word": word,
            "Total Count": total,
            "P(Entailment|Word)": f"{prob_entailment:.1f}%",
            "P(Neutral|Word)": f"{prob_neutral:.1f}%",
            "P(Contradiction|Word)": f"{prob_contradiction:.1f}%"
        })

    return pd.DataFrame(results)

def show_top_artifacts_table(df_results, k=10):
    """
    Returns a single DataFrame with the top k artifacts for Entailment, Neutral, and Contradiction
    displayed side-by-side.
    """
    # Convert percentages to floats
    df = df_results.copy()
    cols = ['P(Entailment|Word)', 'P(Neutral|Word)', 'P(Contradiction|Word)']
    for col in cols:
        if df[col].dtype == 'object':
            df[col] = df[col].astype(str).str.rstrip('%').astype(float)

    # Filter for minimum frequency to avoid rare noise
    df = df[df['Total Count'] >= 100]

    def get_top_k(label_col, prefix):
        top = df.sort_values(label_col, ascending=False).head(k).reset_index(drop=True)
        subset = top[['Word', 'Total Count', label_col]].copy()
        subset.columns = [f'{prefix} Word', 'Count', 'Prob (%)']
        return subset

    ent_df = get_top_k('P(Entailment|Word)', 'Entailment')
    neu_df = get_top_k('P(Neutral|Word)', 'Neutral')
    con_df = get_top_k('P(Contradiction|Word)', 'Contradiction')

    combined_df = pd.concat([ent_df, neu_df, con_df], axis=1)

    return combined_df

df_1 = run_statistical_test(n_grams=1)
df_2 = run_statistical_test(n_grams=2)
full_df = pd.concat([df_1, df_2])
table = show_top_artifacts_table(full_df, k=10)
print(table.to_markdown(index=False))

## Ablation Technique: Hypothesis-Only Dataset

This command executes a critical data preparation step for the diagnostic phase. The goal is to create the final dataset for the Hypothesis-Only Ablation experiment. The script copies the entire SNLI training set but systematically removes the Premise from every example (by replacing it with an empty string). This establishes the lower bound of model performance, proving the degree to which the model relies on artifacts found in the hypothesis alone (if accuracy is > 33.3%).

In [ ]:
import os
import datasets


OUTPUT_FILE = "snli_hypothesis_only.json"

def create_blind_dataset():
    print("Loading SNLI dataset...")
    dataset = datasets.load_dataset("snli")

    dataset = dataset.filter(lambda ex: ex['label'] != -1)

    print(f"Creating Hypothesis-Only version ({len(dataset['train'])} examples)...")

    with open(OUTPUT_FILE, 'w') as f:
        for ex in dataset['train']:
            record = {
                "premise": "", # set the premise to an empty string
                "hypothesis": ex['hypothesis'],
                "label": ex['label']
            }
            f.write(json.dumps(record) + "\n")

    print(f"Success! Saved blind dataset to {OUTPUT_FILE}")

create_blind_dataset()

## Dataset Sanity Check: Hypothesis-Only Model

This command executes a rapid test run using the complete ablation dataset we prepared in the previous step to verify that the highly modified JSON dataset (`./snli_hypothesis_only.json`) loads correctly and that the training process initiates without error before committing to the full 3-epoch training run.

In [ ]:
!python3 run.py \
  --do_train \
  --task nli \
  --dataset ./snli_hypothesis_only.json \
  --output_dir ./test_model_hypothesis_only \
  --per_device_train_batch_size 32 \
  --max_steps 10 \
  --overwrite_output_dir

## Model Ablation: Hypothesis-Only Baseline

This phase executes the ablation experiment against the dataset we prepared in the previous step. The goal is to run the Standard SNLI Model (ELECTRA-Small) but intentionally cripple its input by replacing the premise with an empty string. This quantifies the exact degree to which the model relies on artifacts found in the hypothesis alone. The accuracy result for this experiment must be compared to the random baseline (33.3%). If the model scores significantly higher, the necessity for the mitigation strategies in Part II is confirmed.

In [ ]:
!python3 run.py \
  --do_train \
  --do_eval \
  --task nli \
  --dataset ./snli_hypothesis_only.json \
  --output_dir ./model_hypothesis_only \
  --num_train_epochs 3 \
  --max_length 128 \
  --per_device_train_batch_size 32 \
  --save_steps 0 \
  --eval_strategy epoch \
  --logging_steps 100

## Ablation Result Quantification: Hypothesis-Only Evaluation

This block executes the final two evaluations to quantify the impact of the ablation study.

### A. In-Domain Score (SNLI)

The first command measures the final accuracy of the model when predicting the label based solely on artifacts. This score is compared to the random floor (33.3%) to confirm the quantitative existence of shortcuts.

### B. Robustness Check (ANLI)

The second command confirms the complete vulnerability of the artifact-reliant model against the adversarial ANLI benchmark.

In [ ]:
%%time

print("\n Evaluating Hypothesis-Only Model on SNLI")
!python3 run.py \
  --do_eval \
  --task nli \
  --dataset stanfordnlp/snli \
  --model ./model_hypothesis_only \
  --output_dir ./eval_hypothesis_only_snli \
  --per_device_eval_batch_size 32

print("\n Evaluating Hypothesis-Only Model on ANLI")
!python3 run.py \
  --do_eval \
  --task nli \
  --dataset facebook/anli \
  --model ./model_hypothesis_only \
  --output_dir ./eval_hypothesis_only_anli \
  --per_device_eval_batch_size 32

## Data Filtering: Ambiguous Subset Generation

This block executes the core data curation step for the mitigation phase (Part II) to create the final filtered dataset for Ambiguity Learning by isolating the most valuable training instances. The script reads the training dynamics logs, computes the variability (standard deviation of the gold label probability), and selects the top 33% of the most inconsistent examples. These examples form the high-quality Ambiguous Subset (`snli_ambiguous_only.json`), which is used to retrain the model.

In [ ]:
from scipy.special import softmax
import pandas as pd
import numpy as np
import math
import datasets
from typing import List
from collections import defaultdict
import json

NUM_EPOCHS = 6

def get_ambiguous_instance_ids() -> List[int]:
    guid_to_gold_probabilities = defaultdict(list)

    for epoch in range(1, NUM_EPOCHS + 1):
        file_path = F'model_baseline/dynamics_epoch_{epoch}.jsonl'

        with open(file_path, 'r') as f:
            for line in f:
                data = json.loads(line)
                guid = int(data['guid'])
                logits = data['logits']
                gold = data['gold']
                probs = softmax(logits)
                gold_prob = probs[gold]
                guid_to_gold_probabilities[guid].append(gold_prob)

    guid_to_variability = defaultdict(float)

    for key, value in guid_to_gold_probabilities.items():
        guid_to_variability[key] = np.std(value, dtype=np.float64)

    sorted_by_variability = sorted(guid_to_variability.items(), key=lambda item: item[1])
    sorted_by_variability = dict(sorted_by_variability)

    total_items = len(sorted_by_variability)
    n_largest = math.ceil(total_items * 0.33) # Calculate 33% of the total items

    if n_largest == 0:
        return []

    top_keys = sorted(sorted_by_variability, key=sorted_by_variability.get, reverse=True)[:n_largest]

    return top_keys

ambiguous_ids = set(get_ambiguous_instance_ids())
print(f"Found {len(ambiguous_ids)} ambiguous examples.")


dataset = datasets.load_dataset("snli")
dataset = dataset.filter(lambda ex: ex['label'] != -1)

with open("snli_ambiguous_only.json", "w") as f:
    for i, ex in enumerate(dataset['train']):
        if i in ambiguous_ids:
            record = {
                "premise": ex['premise'],
                "hypothesis": ex['hypothesis'],
                "label": ex['label']
            }
            f.write(json.dumps(record) + "\n")

print("Saved snli_ambiguous_only.json")

## Dataset Generation: Hard Contrast Mining

This script implements the Hard Negative Mining strategy to generate the training datasets for the mitigation experiments to identify hard training examples where simple lexical heuristics fail.

### Methodology
The script filters the source data (both the Ambiguous Subset and the Full SNLI) to find pairs of hypotheses that:
1. Share the same premise.
2. Have high lexical overlap (Jaccard Similarity > 0.5), making them look similar to a bag-of-words model.
3. Have different labels (Contrast), forcing the model to learn the specific semantic difference.

In [ ]:
import json
from datasets import load_dataset
from tqdm import tqdm

def jaccard_similarity_sentences(sentence1, sentence2):
    words1 = set(sentence1.lower().split())
    words2 = set(sentence2.lower().split())
    intersection = len(words1.intersection(words2))
    union = len(words1.union(words2))
    if union == 0: return 0.0
    return intersection / union

def process_and_save_contrasts(examples, output_filename):
    print(f"\nProcessing {len(examples)} examples for {output_filename}...")

    # Group by Premise
    premise_groups = {}
    for ex in examples:
        p = ex['premise']
        if p not in premise_groups:
            premise_groups[p] = []
        premise_groups[p].append(ex)

    # Find Hard Pairs
    hard_examples = []
    for p, group in tqdm(premise_groups.items()):
        if len(group) < 2: continue

        for i in range(len(group)):
            for j in range(i + 1, len(group)):
                ex1 = group[i]
                ex2 = group[j]

                if ex1['label'] == ex2['label']:
                    continue

                sim = jaccard_similarity_sentences(ex1['hypothesis'], ex2['hypothesis'])
                if sim > 0.5:
                    hard_examples.append(ex1)
                    hard_examples.append(ex2)

    unique_data = {json.dumps(ex, sort_keys=True) for ex in hard_examples}
    print(f"Found {len(unique_data)} unique examples for 'Hard Mode'.")

    with open(output_filename, 'w') as f:
        for line in unique_data:
            f.write(line + "\n")
    print(f"Saved to {output_filename}")

print("--- Loading Ambiguous Data ---")
ambiguous_data = []
with open("snli_ambiguous_only.json", "r") as f:
    for line in f:
        ambiguous_data.append(json.loads(line))

process_and_save_contrasts(ambiguous_data, "snli_ambiguous_contrast_pairs.json")

print("\n--- Loading Full SNLI Data ---")
dataset = load_dataset("snli", split="train")
full_data = [ex for ex in dataset if ex['label'] != -1]

process_and_save_contrasts(full_data, "snli_full_contrast_pairs.json")

## Dataset Sanity Check: Ambiguous Contrast Model

This command performs a rapid test run using the newly generated Ambiguous Contrast dataset (the filtered subset) to verify that the custom JSON dataset (`snli_ambiguous_contrast_pairs.json`) loads correctly and that the training loop initiates without errors before committing to the full 10-epoch experiment.

In [ ]:
!python3 run.py \
  --do_train \
  --task nli \
  --dataset ./snli_ambiguous_contrast_pairs.json \
  --output_dir ./test_model_ambiguous_contrast \
  --per_device_train_batch_size 32 \
  --max_steps 10 \
  --overwrite_output_dir

## Dataset Sanity Check: Full Contrast Model

This command performs a rapid test run using the newly generated Full Contrast dataset to verify that the custom JSON dataset (`snli_full_contrast_pairs.json`) loads correctly and the training process initiates without errors before committing to the long-running experiment.

In [ ]:
!python3 run.py \
  --do_train \
  --task nli \
  --dataset ./snli_full_contrast_pairs.json \
  --output_dir ./test_model_full_contrast \
  --per_device_train_batch_size 32 \
  --max_steps 10 \
  --overwrite_output_dir

## Mitigation Experiment: Ambiguous Contrast Training

This command executes the core mitigation strategy by training the model exclusively on the highly curated and filtered dataset to test the central hypothesis that Data Pruning (removing low-variability artifacts) and Hard Negative Mining (Contrastive Pairs) yields a model more robust to adversarial examples than the Standard Baseline.

In [ ]:
%%time

!python3 run.py \
  --do_train \
  --do_eval \
  --task nli \
  --dataset ./snli_ambiguous_contrast_pairs.json \
  --output_dir ./model_ambiguous_contrast \
  --num_train_epochs 10 \
  --max_length 128 \
  --per_device_train_batch_size 32 \
  --eval_strategy epoch \
  --save_strategy epoch \
  --logging_steps 100 \
  --load_best_model_at_end \
  --save_total_limit 2

## Mitigation Experiment: Full Contrast Training

This command executes the second mitigation experiment, training the model on the largest possible dataset filtered only for hard negative pairs to test the hypothesis that Hard Negative Mining alone (without initial Ambiguity Filtering) is sufficient to improve robustness, serving as a direct comparison against the Ambiguous Contrast model.

In [ ]:
%%time

!python3 run.py \
  --do_train \
  --do_eval \
  --task nli \
  --dataset ./snli_full_contrast_pairs.json \
  --output_dir ./model_full_contrast \
  --num_train_epochs 6 \
  --max_length 128 \
  --per_device_train_batch_size 32 \
  --eval_strategy epoch \
  --save_strategy epoch \
  --logging_steps 100 \
  --load_best_model_at_end \
  --save_total_limit 2

## Evaluation: Ambiguous Contrast Model

This command executes the evaluation for the Ambiguous Contrast Model.

### A. Robustness Check (ANLI)

The first command quantifies the success of the Ambiguity Filtering + Hard Negative Mining strategy. This score is the central result that tests the hypothesis that data filtering improves OOD robustness.

### B. In-Domain Accuracy Check (SNLI)

The second command verifies the trade-off.

In [ ]:
%%time

print("\n Evaluating Ambiguous Contrast Model on ANLI")
!python3 run.py \
  --do_eval \
  --task nli \
  --dataset facebook/anli \
  --model ./model_ambiguous_contrast \
  --output_dir ./eval_ambiguous_contrast_anli

print("\n Evaluating Ambiguous Contrast Model on SNLI")
!python3 run.py \
  --do_eval \
  --task nli \
  --dataset stanfordnlp/snli \
  --model ./model_ambiguous_contrast \
  --output_dir ./eval_ambiguous_contrast_snli

## Evaluation: Full Contrast Model

This command executes the evaluation for the Full Contrast Model.

### A. Robustness Check (ANLI)

The first command quantifies the final performance of the Hard Contrast Mining method when applied to the entire, unpruned SNLI dataset. This score is compared directly against the Ambiguous Contrast model (previous step) to isolate the value of Data Pruning.

### B. In-Domain Accuracy Check (SNLI)

The second command determines the ceiling performance for the mitigation strategy. Since this model trained on significantly more data than the ambiguous model, it is expected to achieve a higher overall in-domain score, even if its robustness is lower.

In [ ]:
%%time

print("\n Evaluating Full Contrast Model on ANLI")
!python3 run.py \
  --do_eval \
  --task nli \
  --dataset facebook/anli \
  --model ./model_full_contrast \
  --output_dir ./eval_full_contrast_anli

print("\n Evaluating Full Contrast Model on SNLI")
!python3 run.py \
  --do_eval \
  --task nli \
  --dataset stanfordnlp/snli \
  --model ./model_full_contrast \
  --output_dir ./eval_full_contrast_snli

## Data Augmentation Generation: Augmented Ambiguous Contrast

This block executes the creation of the final Augmented Ambiguous Contrast dataset to address the data scarcity of the Ambiguous Contrast dataset (~8k examples) by artificially increasing its size through high-quality back-translation (English-Spanish-English). The resulting dataset increases the volume of high-variability data without re-introducing Easy artifacts.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
from sentence_transformers import SentenceTransformer, util
import torch
import json
from tqdm import tqdm

SIMILARITY_THRESHOLD = 0.85
BATCH_SIZE = 32
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def is_similar(model, text1, text2):
  # Reject if exactly the same (duplicate)
  if text1.strip().lower() == text2.strip().lower():
    return False

  emb1 = model.encode(text1, convert_to_tensor=True)
  emb2 = model.encode(text2, convert_to_tensor=True)
  score = util.cos_sim(emb1, emb2).item()
  return score > SIMILARITY_THRESHOLD

def get_back_translated_text(texts, en_es_tokenizer, en_es_model, es_en_tokenizer, es_en_model):
  spanish_translated = translate(texts, en_es_model, en_es_tokenizer)
  back_translated_text = translate(spanish_translated, es_en_model, es_en_tokenizer)
  return back_translated_text

def translate(texts, model, tokenizer):
  inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
  with torch.no_grad():
      translated_tokens = model.generate(**inputs)
  decoded = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)
  return decoded

def process_and_save_augmented_contrasts(examples, output_filename):
  print(f"\nProcessing {len(examples)} examples for {output_filename}...")

  print("Loading SBERT model...")
  similarity_model = SentenceTransformer('all-MiniLM-L6-v2').to(device)

  print("Loading Translation models...")
  # English to Spanish
  en_es_model_name = 'Helsinki-NLP/opus-mt-en-es'
  en_es_tokenizer = MarianTokenizer.from_pretrained(en_es_model_name)
  en_es_model = MarianMTModel.from_pretrained(en_es_model_name).to(device)

  # Spanish to English
  es_en_model_name = 'Helsinki-NLP/opus-mt-es-en'
  es_en_tokenizer = MarianTokenizer.from_pretrained(es_en_model_name)
  es_en_model = MarianMTModel.from_pretrained(es_en_model_name).to(device)

  augmented_examples = []

  for i in tqdm(range(0, len(examples), BATCH_SIZE)):
    batch = examples[i : i + BATCH_SIZE]

    batch_premises = [ex['premise'] for ex in batch]
    batch_hypotheses = [ex['hypothesis'] for ex in batch]

    trans_premises = get_back_translated_text(
        batch_premises, en_es_tokenizer, en_es_model, es_en_tokenizer, es_en_model
    )
    trans_hypotheses = get_back_translated_text(
        batch_hypotheses, en_es_tokenizer, en_es_model, es_en_tokenizer, es_en_model
    )

    for j, ex in enumerate(batch):
      # Always keep original
      augmented_examples.append(ex)

      orig_prem = batch_premises[j]
      orig_hyp = batch_hypotheses[j]
      new_prem = trans_premises[j]
      new_hyp = trans_hypotheses[j]

      valid_prem = is_similar(similarity_model, orig_prem, new_prem)
      final_prem = new_prem if valid_prem else orig_prem

      valid_hyp = is_similar(similarity_model, orig_hyp, new_hyp)
      final_hyp = new_hyp if valid_hyp else orig_hyp

      if valid_prem or valid_hyp:
        augmented_examples.append({
            "premise": final_prem,
            "hypothesis": final_hyp,
            "label": ex['label']
        })

  unique_data = {json.dumps(ex, sort_keys=True) for ex in augmented_examples}
  print(f"Found {len(unique_data)} unique examples.")

  with open(output_filename, 'w') as f:
      for line in unique_data:
          f.write(line + "\n")
  print(f"Saved to {output_filename}")

print("--- Loading Ambiguous Contrast Data ---")
ambiguous_data = []
try:
    with open("snli_ambiguous_contrast_pairs.json", "r") as f:
        for line in f:
            ambiguous_data.append(json.loads(line))

    process_and_save_augmented_contrasts(ambiguous_data, "snli_augmented_ambiguous_contrast_pairs.json")
except FileNotFoundError:
    print("Error: Input file not found.")

## Dataset Sanity Check: Augmented Ambiguous Contrast

This command executes a rapid test run using the Augmented Ambiguous Contrast dataset to verify that the highly processed data file (`snli_augmented_ambiguous_contrast_pairs.json`), which includes both filtering and back-translation, loads correctly and that the large file size does not cause the training environment to crash.

In [ ]:
!python3 run.py \
  --do_train \
  --task nli \
  --dataset ./snli_augmented_ambiguous_contrast_pairs.json \
  --output_dir ./test_augmented_ambiguous_contrast \
  --per_device_train_batch_size 32 \
  --max_steps 10 \
  --overwrite_output_dir

## Experiment: Augmented Ambiguous Contrast Training

This command executes the final and most comprehensive mitigation strategy, training the model on the refined and augmented dataset to combine the benefits of Ambiguity Filtering and Data Augmentation, aiming for the highest robustness score on the ANLI benchmark. This test determines if addressing data scarcity (via augmentation) further improves the robustness gains achieved through filtering.

In [ ]:
%%time

!python3 run.py \
  --do_train \
  --do_eval \
  --task nli \
  --dataset ./snli_augmented_ambiguous_contrast_pairs.json \
  --output_dir ./model_augmented_ambiguous_contrast \
  --num_train_epochs 10 \
  --max_length 128 \
  --per_device_train_batch_size 32 \
  --eval_strategy epoch \
  --save_strategy epoch \
  --logging_steps 100 \
  --load_best_model_at_end \
  --save_total_limit 2

## Evaluation: Augmented Ambiguous Contrast Model

This block executes the final evaluations to completely assess the mitigation strategy's performance.

### Robustness Check (ANLI)

The first command determines if the added diversity from Data Augmentation pushed the ANLI robustness score beyond the non-augmented Ambiguous Contrast model (34.5%).

### In-Domain Accuracy Check (SNLI)

The second command verifies that the high validation accuracy seen during training holds on the final SNLI test set. A high score here confirms the success of the augmentation in solving data scarcity without relearning artifacts.

In [ ]:
%%time

print("\n Evaluating Augmented Ambiguous Contrast Model on ANLI")
!python3 run.py \
  --do_eval \
  --task nli \
  --dataset facebook/anli \
  --model ./model_augmented_ambiguous_contrast \
  --output_dir ./eval_augmented_ambiguous_contrast_anli

print("\n Evaluating Augmented Ambiguous Contrast Model on SNLI")
!python3 run.py \
  --do_eval \
  --task nli \
  --dataset stanfordnlp/snli \
  --model ./model_augmented_ambiguous_contrast \
  --output_dir ./eval_augmented_ambiguous_contrast_snli

## Data Augmentation Generation: Multilingual Back-Translation

This script executes the **Multilingual Augmentation** experiment. It extends the back-translation pipeline to use multiple pivot languages (Spanish, German, and French) in an attempt to maximize lexical and syntactic diversity.

### Methodology
1. **Round-Trip Translation:** Iterates through the list `['es', 'de', 'fr']`. For each language, it uses `Helsinki-NLP/MarianMT` to translate the premise and hypothesis from English to the target language and back.
2. **Semantic Filtering:** Uses `SentenceTransformer` (SBERT) to validate the quality of the paraphrase.
    * **Similarity Threshold:** `0.85` (Ensures meaning is preserved).
    * **Identity Check:** Discards the result if it matches the original string exactly.
3. **Aggregation:** New valid examples are added to a `master_set` dictionary to automatically handle deduplication across different languages.
4. **Resource Management:** Includes explicit garbage collection and CUDA cache clearing between languages to prevent Out-Of-Memory (OOM) errors on the T4 GPU.

In [ ]:
import json
import torch
from transformers import MarianMTModel, MarianTokenizer
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm
import gc
import os

SIMILARITY_THRESHOLD = 0.85
BATCH_SIZE = 16
LANGUAGES = ["es", "de", "fr"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
INPUT_FILE = "snli_ambiguous_contrast_pairs.json"
FINAL_OUTPUT_FILE = "snli_multilingual_ambiguous_contrast_pairs.json"

print(f"Using device: {DEVICE}")

def batch_translate(texts, tokenizer, model):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(DEVICE)
    with torch.no_grad():
        translated = model.generate(**inputs)
    return tokenizer.batch_decode(translated, skip_special_tokens=True)

def is_valid(model, original, augmented):
    if original.strip().lower() == augmented.strip().lower(): return False
    emb1 = model.encode(original, convert_to_tensor=True, device=DEVICE)
    emb2 = model.encode(augmented, convert_to_tensor=True, device=DEVICE)
    return util.cos_sim(emb1, emb2).item() > SIMILARITY_THRESHOLD

def augment_single_language(data, lang, sbert):
    output_filename = f"temp_aug_{lang}.json"

    if os.path.exists(output_filename):
        print(f"Found existing file for {lang}, skipping generation.")
        with open(output_filename, 'r') as f:
            return [json.loads(line) for line in f]

    print(f"\n=== Augmenting with Language: {lang.upper()} ===")

    model_name_fwd = f'Helsinki-NLP/opus-mt-en-{lang}'
    model_name_back = f'Helsinki-NLP/opus-mt-{lang}-en'

    tok_fwd = MarianTokenizer.from_pretrained(model_name_fwd)
    mod_fwd = MarianMTModel.from_pretrained(model_name_fwd).to(DEVICE)
    tok_back = MarianTokenizer.from_pretrained(model_name_back)
    mod_back = MarianMTModel.from_pretrained(model_name_back).to(DEVICE)

    new_examples = []

    for i in tqdm(range(0, len(data), BATCH_SIZE)):
        batch = data[i : i + BATCH_SIZE]
        orig_prems = [ex['premise'] for ex in batch]
        orig_hyps = [ex['hypothesis'] for ex in batch]

        try:
            mid_prems = batch_translate(orig_prems, tok_fwd, mod_fwd)
            mid_hyps = batch_translate(orig_hyps, tok_fwd, mod_fwd)
            new_prems = batch_translate(mid_prems, tok_back, mod_back)
            new_hyps = batch_translate(mid_hyps, tok_back, mod_back)
        except Exception as e:
            print(f"Batch failed: {e}")
            continue

        for j, ex in enumerate(batch):
            valid_p = is_valid(sbert, orig_prems[j], new_prems[j])
            final_prem = new_prems[j] if valid_p else orig_prems[j]

            valid_h = is_valid(sbert, orig_hyps[j], new_hyps[j])
            final_hyp = new_hyps[j] if valid_h else orig_hyps[j]

            if valid_p or valid_h:
                new_examples.append({
                    "premise": final_prem,
                    "hypothesis": final_hyp,
                    "label": ex['label']
                })

    # Save intermediate file
    with open(output_filename, 'w') as f:
        for ex in new_examples:
            f.write(json.dumps(ex) + "\n")

    # Cleanup to free memory
    del tok_fwd, mod_fwd, tok_back, mod_back
    torch.cuda.empty_cache()
    gc.collect()

    return new_examples

def run_multilingual_augmentation():
    print(f"Loading {INPUT_FILE}...")
    with open(INPUT_FILE, 'r') as f:
        originals = [json.loads(line) for line in f]

    master_set = {}
    for ex in originals:
        key = (ex['premise'], ex['hypothesis'], ex['label'])
        master_set[key] = ex

    print("Loading SBERT...")
    sbert = SentenceTransformer('all-MiniLM-L6-v2').to(DEVICE)

    for lang in LANGUAGES:
        aug_data = augment_single_language(originals, lang, sbert)

        for ex in aug_data:
            key = (ex['premise'], ex['hypothesis'], ex['label'])
            master_set[key] = ex

    final_data = list(master_set.values())
    print(f"\nFinal Dataset Size: {len(final_data)}")

    with open(FINAL_OUTPUT_FILE, 'w') as f:
        for ex in final_data:
            f.write(json.dumps(ex) + "\n")
    print(f"Saved to {FINAL_OUTPUT_FILE}")

run_multilingual_augmentation()

## Dataset Sanity Check: Multilingual Augmented Model

This command performs a rapid test run using the newly generated Multilingual Augmented dataset. It verifies that the custom JSON file (`snli_multilingual_ambiguous_contrast_pairs.json`) loads correctly and that the training process initiates without error before committing to the full experiment.

In [ ]:
!python3 run.py \
  --do_train \
  --task nli \
  --dataset ./snli_multilingual_ambiguous_contrast_pairs.json \
  --output_dir ./test_multilingual_ambiguous_contrast \
  --per_device_train_batch_size 32 \
  --max_steps 10 \
  --overwrite_output_dir

## Mitigation Experiment: Multilingual Augmented Training

This command executes the multilingual variation of the mitigation strategy. It trains the model on the dataset augmented with Spanish, German, and French back-translations. The objective is to test if increasing linguistic diversity through multiple pivot languages yields better robustness than the single-language Spanish approach.

In [ ]:
%%time

!python3 run.py \
  --do_train \
  --do_eval \
  --task nli \
  --dataset ./snli_multilingual_ambiguous_contrast_pairs.json \
  --output_dir ./model_multilingual_ambiguous_contrast \
  --num_train_epochs 10 \
  --max_length 128 \
  --per_device_train_batch_size 32 \
  --eval_strategy epoch \
  --save_strategy epoch \
  --logging_steps 100 \
  --load_best_model_at_end \
  --save_total_limit 2

## Evaluation: Multilingual Augmented Model

This block executes the final evaluations to assess the performance of the Multilingual Augmented model. These results are compared directly against the Spanish-only augmented model to determine if adding more languages improved robustness or introduced noise (semantic drift).

### Robustness Check (ANLI)

The first command evaluates the model on the adversarial benchmark. This score reveals whether the increased syntactic diversity from German and French translates into better reasoning capabilities or if it creates a trade-off with accuracy.

### In-Domain Accuracy Check (SNLI)

The second command measures the model's performance on the standard test set. This checks if the multilingual augmentation preserved the core logic of the task or if the additional languages degraded the model's ability to handle standard examples.

In [ ]:
%%time

print("\n Evaluating Multilingual Augmented Ambiguous Contrast Model on ANLI")
!python3 run.py \
  --do_eval \
  --task nli \
  --dataset facebook/anli \
  --model ./model_multilingual_ambiguous_contrast \
  --output_dir ./eval_multilingual_ambiguous_contrast_anli

print("\n Evaluating Multilingual Augmented Ambiguous Contrast Model on SNLI")
!python3 run.py \
  --do_eval \
  --task nli \
  --dataset stanfordnlp/snli \
  --model ./model_multilingual_ambiguous_contrast \
  --output_dir ./eval_multilingual_ambiguous_contrast_snli

## Final Visualization: Performance Comparison

This block generates the summary visualization for the project report. It aggregates the final accuracy scores from all experiments (Baseline, Ablation, and Mitigation) into a single grouped bar chart to visualize the trade-off between in-domain performance and adversarial robustness.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

data = {
    'Model': [
        'Baseline (Full SNLI)',
        'Hypothesis-Only',
        'Full Contrast',
        'Ambiguous Contrast',
        'Augmented (En-Es)',
        'Multilingual (En-Es/De/Fr)'
    ],
    'SNLI Accuracy': [90.1, 60.0, 84.5, 70.0, 70.1, 70.7],
    'ANLI Accuracy': [32.6, 33.8, 31.1, 34.5, 34.9, 34.0]
}

df = pd.DataFrame(data)

df_melted = df.melt(id_vars="Model", var_name="Dataset", value_name="Accuracy (%)")

plt.figure(figsize=(12, 7))
sns.set_theme(style="whitegrid")

ax = sns.barplot(
    data=df_melted,
    x="Model",
    y="Accuracy (%)",
    hue="Dataset",
    palette=["#a1c9f4", "#ff9f9b"] # Light Blue (SNLI), Light Red (ANLI)
)


plt.title("Project Summary: Impact of Data Strategies on Robustness", fontsize=16, pad=20)
plt.xlabel("")
plt.ylabel("Accuracy (%)", fontsize=12)
plt.xticks(rotation=15, ha='right')
plt.ylim(25, 100)

# Add Random Baseline Line (33.3%)
plt.axhline(y=33.3, color='grey', linestyle='--', linewidth=1.5, label='Random Guessing (33.3%)')
plt.legend(loc='upper right')


for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.text(p.get_x() + p.get_width() / 2., height + 1,
                f'{height:.1f}%',
                ha="center", fontsize=10, fontweight='bold', color='black')

plt.tight_layout()

plt.savefig('final_results_comparison.png', dpi=300)
print("Plot saved as final_results_comparison.png")
plt.show()